# Batch decisions with datasets.map

Executed on CPU with the real checkpoint. Uses public FLEURS validation audio at a pinned revision. Audio and transcripts are not embedded in outputs. No performance or accuracy estimate is derived from these two rows. Decoder context above eight seconds is **unvalidated**.

Install from a checkout with `pip install '.[torch,datasets]'` and compatible TorchCodec/FFmpeg, plus `nbclient nbformat ipykernel`. Authenticate outside this notebook if required. Converted head/readout weights are Apache-2.0.


In [1]:
import torch
from datasets import Audio, load_dataset

from marimba import Marimba, Question

torch.set_num_threads(4)
revision = "70bb2e84b976b7e960aa89f1c648e09c59f894dd"
url = (
    "hf://datasets/google/fleurs@"
    + revision
    + "/parquet-data/en_us/validation-00000-of-00001.parquet"
)
ds = load_dataset("parquet", data_files={"validation": url}, split="validation[:2]")
ds = ds.cast_column("audio", Audio(sampling_rate=16000))
model = Marimba.from_pretrained(
    revision="bc980b4e6eb34e7fea96ea8b0884c2093a845dc5",
    backend="torch",
    device="cpu",
    dtype="float32",
)
print(
    {
        "dataset": "google/fleurs",
        "revision": revision,
        "split": "validation",
        "rows": len(ds),
        "backend": model.runtime.backend,
        "device": model.runtime.device,
        "dtype": model.runtime.dtype,
    }
)

{'dataset': 'google/fleurs', 'revision': '70bb2e84b976b7e960aa89f1c648e09c59f894dd', 'split': 'validation', 'rows': 2, 'backend': 'torch', 'device': 'cpu', 'dtype': 'float32'}


In [2]:
questions = [Question.noul("Is someone speaking?")]
scored = ds.map(model.map_fn(questions), batched=True, batch_size=2, load_from_cache_file=False)
probabilities = [round(row["probability"], 6) for row in scored["marimba_0"]]
print({"rows": len(scored), "decision_column": "marimba_0", "probabilities": probabilities})
assert len(scored) == 2 and all(p is not None for p in probabilities)

{'rows': 2, 'decision_column': 'marimba_0', 'probabilities': [0.843403, 0.976881]}
